# Setting up login, tables names and file paths

In [0]:
dbutils.widgets.text("login", "", "Enter your login")
dbutils.widgets.text("catalog", "dbr_dev", "Enter your catalog")

In [0]:
login = dbutils.widgets.get("login")
catalog = dbutils.widgets.get("catalog")

source_data_name = "alabama_sold_real_estate"
source_schema:str = login
target_schema = f"{login}_bronze"


source_table_name = f"{catalog}.{login}.{source_data_name}_intelligence_2026"
target_table_name = f"{catalog}.{login}_bronze.{source_data_name}_bronze"

# Proper workflow

## Reading the metadata from the source table

In [0]:
from pyspark.sql.functions import current_timestamp, col
raw_df = spark.read.table(source_table_name)

updates_raw_df = (raw_df
  .withColumn("ingestion_timestamp", current_timestamp())
  .withColumn("source_filename", col("_metadata.file_path"))

)

In [0]:
target_table_exists = spark.catalog.tableExists(target_table_name)

if target_table_exists:
    from delta.tables import DeltaTable
    print(f"Target table already exists")

    bronze_table = DeltaTable.forName(spark, target_table_name)

  
  # If the target table doesn't have the columns we want to add, add them.
    bronze_table.toDF().createOrReplaceTempView("bronze_temp_view")
    spark.sql(f"""
        ALTER TABLE {target_table_name}
        ADD COLUMNS (
            ingestion_timestamp TIMESTAMP,
            source_filename STRING
        )
    """
    )

    (bronze_table.alias("target")
     .merge(
         updates_raw_df.alias("source"),
         "target.Index = source.Index"
     )
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute()
    )
else:
    updates_raw_df.write.mode("overwrite").saveAsTable(target_table_name)